# Global Comparison of Eight Window Lengths

Within-user accuracy, macro-F1, and macro-AUC across eight window lengths and three participants.

Exploratory findings require within-participant exchangeability. Unequal test-set sizes, same-session dependence, and the original preprocessing limit inference.


In [1]:
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from IPython.display import display, Markdown

SEED = 42
N_PERMUTATIONS = 199_999
HOLM_FAMILY_SIZE = 36
ALPHA = 0.05
WINDOWS_S = [0.01, 0.02, 0.05, 0.15, 0.25, 0.50, 1.00, 2.00]
USERS = ["0007", "0008", "0010"]
METRICS = ["accuracy", "f1_macro", "auc_macro"]
LABELS = {"accuracy": "Accuracy", "f1_macro": "Macro-F1", "auc_macro": "Macro-AUC"}
RESULT_FOLDERS = {"MLP": "Results-Data-ML-1-v4", "CNN1D": "Results-Data-ML-2-v4"}

sys.dont_write_bytecode = True
random.seed(SEED)
np.random.seed(SEED)
REPO = next((p.resolve() for p in [Path.cwd(), Path.cwd() / "repo"]
             if (p / "Outputs").is_dir()), None)
if REPO is None:
    raise FileNotFoundError("Run from the repository or its parent directory.")

## Inputs


In [2]:
rows = []
for model, dirname in RESULT_FOLDERS.items():
    for window in WINDOWS_S:
        parent = REPO / "Outputs" / dirname / f"ml_win_{window:.2f}s" / "by_user"
        assert sorted(p.name.removeprefix("user_") for p in parent.glob("user_*")) == USERS
        for user in USERS:
            folder = parent / f"user_{user}"
            classes = json.loads((folder / "classes.json").read_text(encoding="utf-8"))["classes"]
            stored = json.loads((folder / "metrics.json").read_text(encoding="utf-8"))
            assert classes == stored["classes"] == [f"g{i}" for i in range(1, 7)]
            y = pd.read_csv(folder / "arrays" / "y_true.csv")["y_true"].to_numpy()
            pred = pd.read_csv(folder / "arrays" / "y_pred.csv")["y_pred"].to_numpy()
            probabilities = pd.read_csv(folder / "arrays" / "y_prob.csv")
            assert list(probabilities.columns) == [f"p_{c}" for c in classes]
            p = probabilities.to_numpy(dtype=float)
            assert len(y) == len(pred) == len(p) == stored["test_shape"][0]
            assert np.array_equal(np.unique(y), np.arange(6))
            assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
            assert np.allclose(p.sum(axis=1), 1, atol=1e-6, rtol=0)
            assert np.array_equal(pred, p.argmax(axis=1))
            values = {
                "accuracy": accuracy_score(y, pred),
                "f1_macro": f1_score(y, pred, labels=np.arange(6), average="macro", zero_division=0),
                "auc_macro": np.mean([roc_auc_score(y == i, p[:, i]) for i in range(6)]),
            }
            for metric in ("accuracy", "f1_macro"):
                assert abs(values[metric] - stored["metrics"][metric]) <= 0.000050001
            rows.append({"model": model, "window_s": window, "user": user, **values})

metrics = pd.DataFrame(rows)
assert len(metrics) == 48
assert not metrics.duplicated(["model", "window_s", "user"]).any()
assert np.isfinite(metrics[METRICS].to_numpy()).all()

## Global Tests
Tie-corrected Friedman statistic with 199,999 within-participant permutations, seed 42, and Monte Carlo +1 correction. Holm uses the original 36-test family, with p = 1 assigned conservatively to the 30 omitted comparisons.


In [3]:
def permuted_friedman(values, seed):
    n, k = values.shape
    ranks = stats.rankdata(values, axis=1, method="average")
    ties = sum(sum(t ** 3 - t for t in np.unique(row, return_counts=True)[1])
               for row in values)
    correction = 1.0 - ties / (n * (k ** 3 - k))

    def statistic(r):
        if correction <= 1e-15:
            return np.zeros(r.shape[:-2])
        q = (12.0 / (n * k * (k + 1)) * np.square(r.sum(axis=-2)).sum(axis=-1)
             - 3.0 * n * (k + 1)) / correction
        return np.maximum(q, 0)

    observed = float(statistic(ranks))
    if correction <= 1e-15:
        return observed, 1.0
    assert np.isclose(observed, stats.friedmanchisquare(*values.T).statistic)
    rng = np.random.default_rng(seed)
    exceedances = 0
    for start in range(0, N_PERMUTATIONS, 4096):
        count = min(4096, N_PERMUTATIONS - start)
        shuffled = rng.permuted(np.broadcast_to(ranks, (count, n, k)), axis=2)
        exceedances += np.count_nonzero(statistic(shuffled) >= observed - 1e-12)
    return observed, (exceedances + 1) / (N_PERMUTATIONS + 1)

def holm_adjust(pvalues, family_size):
    assert family_size >= len(pvalues)
    p = np.concatenate([np.asarray(pvalues), np.ones(family_size - len(pvalues))])
    order = np.argsort(p, kind="stable")
    adjusted = np.empty_like(p)
    adjusted[order] = np.minimum(1.0, np.maximum.accumulate(
        p[order] * np.arange(family_size, 0, -1)))
    return adjusted[:len(pvalues)]

results = []
for model_index, model in enumerate(RESULT_FOLDERS):
    for metric_index, metric in enumerate(METRICS):
        values = metrics.loc[metrics.model == model].pivot(
            index="user", columns="window_s", values=metric).loc[USERS, WINDOWS_S].to_numpy()
        seed = np.random.SeedSequence([SEED, model_index, metric_index])
        q, p = permuted_friedman(values, seed)
        results.append({"Model": model, "Metric": LABELS[metric], "Friedman Q": q, "p": p})

results = pd.DataFrame(results)
results["Holm p"] = holm_adjust(results["p"].to_numpy(), HOLM_FAMILY_SIZE)
results["Holm p < 0.05"] = results["Holm p"] < ALPHA

## Results


In [4]:
display(results.round(6))
significant = results.loc[results["Holm p < 0.05"]]
findings = ", ".join(
    f"{row['Model']} {row['Metric']} (Holm-adjusted p={row['Holm p']:.5f})"
    for _, row in significant.iterrows()
)
summary = (
    f"Exploratory global comparisons across the eight window lengths indicated differences in {findings}. "
    "These findings are conditional on within-participant exchangeability of window conditions."
    if findings else
    "No global comparison rejected after Holm adjustment. Non-rejection does not establish equivalence."
)
display(Markdown(summary))

,Model,Metric,Friedman Q,p,Holm p,Holm p < 0.05
0,MLP,Accuracy,17.725806,0.000425,0.01445,True
1,MLP,Macro-F1,19.444444,0.000020,0.00072,True
2,MLP,Macro-AUC,17.333333,0.000660,0.02178,True
3,CNN1D,Accuracy,12.529412,0.048115,1.00000,False
4,CNN1D,Macro-F1,12.529412,0.048625,1.00000,False
5,CNN1D,Macro-AUC,19.649123,0.000040,0.00140,True


Exploratory global comparisons across the eight window lengths indicated differences in MLP Accuracy (Holm-adjusted p=0.01445), MLP Macro-F1 (Holm-adjusted p=0.00072), MLP Macro-AUC (Holm-adjusted p=0.02178), CNN1D Macro-AUC (Holm-adjusted p=0.00140). These findings are conditional on within-participant exchangeability of window conditions.